##### Setup

###### 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from PIL import Image
from IPython.display import display, FileLink, Image as ImagemNotebook

PASTA = Path('lista7_resultados')
PASTA.mkdir(exist_ok=True)

###### 2.

In [ ]:
def deslizador(valor, minimo, maximo, passo, descricao):
    return widgets.FloatSlider(value=valor, min=minimo, max=maximo, step=passo,
                               description=descricao, continuous_update=False)


def exibir_controles(funcao, controles):
    saida = widgets.interactive_output(funcao, controles)
    display(widgets.VBox(list(controles.values())), saida)
    return saida


def normalizar(v):
    v = np.asarray(v, dtype=float)
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12)


def mostrar_imagens(imagens, titulos, arquivo, legendas=None):
    figura, eixos = plt.subplots(1, len(imagens), figsize=(4*len(imagens), 4.8), squeeze=False)
    for i, (eixo, imagem, titulo) in enumerate(zip(eixos[0], imagens, titulos)):
        eixo.imshow(imagem)
        eixo.set_title(titulo)
        eixo.set_xticks([])
        eixo.set_yticks([])
        if legendas is not None:
            eixo.set_xlabel(legendas[i], fontsize=9)
    figura.tight_layout()
    figura.savefig(PASTA / arquivo, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(figura)

###### 3.

In [ ]:
def construir_elipsoide(aneis=10, segmentos=20):
    # Polos únicos evitam triângulos degenerados e fecham a malha.
    vertices = [[0, 1.15, 0]]
    for i in range(1, aneis):
        theta = np.pi*i/aneis
        for j in range(segmentos):
            phi = 2*np.pi*j/segmentos
            vertices.append([0.85*np.sin(theta)*np.cos(phi),
                             1.15*np.cos(theta), 0.65*np.sin(theta)*np.sin(phi)])
    vertices.append([0, -1.15, 0])
    faces = []
    for j in range(segmentos):
        faces.append([0, 1+j, 1+(j+1)%segmentos])
    for i in range(aneis-2):
        for j in range(segmentos):
            a = 1+i*segmentos+j
            b = 1+i*segmentos+(j+1)%segmentos
            faces.extend([[a, a+segmentos, b], [b, a+segmentos, b+segmentos]])
    ultimo = len(vertices)-1
    for j in range(segmentos):
        faces.append([ultimo, ultimo-segmentos+(j+1)%segmentos, ultimo-segmentos+j])
    vertices = np.array(vertices, float)
    faces = np.array(faces, int)
    # Corrige a orientação das faces usando o centro do elipsoide convexo.
    triangulos = vertices[faces]
    normais_faces = np.cross(triangulos[:, 1]-triangulos[:, 0], triangulos[:, 2]-triangulos[:, 0])
    inverter = np.sum(normais_faces*triangulos.mean(axis=1), axis=1) < 0
    faces[inverter] = faces[inverter][:, [0, 2, 1]]
    normais = normalizar(vertices / np.array([0.85, 1.15, 0.65])**2)
    return vertices, faces, normais


VERTICES, FACES, NORMAIS = construir_elipsoide()


def rotacao(angulo, eixo):
    a = np.deg2rad(angulo)
    c, s = np.cos(a), np.sin(a)
    if eixo == 'Y':
        return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def estado(t, voltas=1, ks_inicial=0.02, ks_final=0.95, s_inicial=5, s_final=70):
    # Smoothstep mantém a transição de material contínua nas extremidades.
    u = t*t*(3-2*t)
    return 360*voltas*t, (1-u)*ks_inicial+u*ks_final, (1-u)*s_inicial+u*s_final


def renderizar(t=0, resolucao=160, voltas=1, ks=None, expoente=None,
               componente=3, luz_x=3, luz_y=3, luz_z=4, camera=4.5,
               projecao='Perspectiva', luz_movel=False, inclinacao=18,
               ks_inicial=0.02, ks_final=0.95, s_inicial=5, s_final=70):
    angulo, ks_t, s_t = estado(t, voltas, ks_inicial, ks_final, s_inicial, s_final)
    ks = ks_t if ks is None else ks
    expoente = s_t if expoente is None else expoente
    matriz = rotacao(angulo, 'Y') @ rotacao(inclinacao, 'X')
    pontos = VERTICES @ matriz.T
    normais = NORMAIS @ matriz.T
    olho = np.array([0, 0, camera])
    luz = np.array([luz_x, luz_y, luz_z], float)
    if luz_movel:
        luz = rotacao(360*t, 'Y') @ luz
    profundidades = camera-pontos[:, 2]
    # Câmera em +Z olhando para a origem; plano próximo=0.1, distante=20.
    escala = 2.7/profundidades if projecao == 'Perspectiva' else np.full(len(pontos), 0.62)
    tela = (pontos[:, :2]*escala[:, None]*np.array([1, -1])+1)*(resolucao-1)/2
    framebuffer = np.full((resolucao, resolucao, 3), [0.035, 0.045, 0.075])
    zbuffer = np.full((resolucao, resolucao), np.inf)
    cor = np.array([0.12, 0.54, 0.78])
    for face in FACES:
        z = profundidades[face]
        if np.any((z < 0.1) | (z > 20)):
            continue
        a, b, c = tela[face]
        matriz_2d = np.column_stack((b-a, c-a))
        if abs(np.linalg.det(matriz_2d)) < 1e-9:
            continue
        minimo = np.maximum(np.floor(tela[face].min(axis=0)).astype(int), 0)
        maximo = np.minimum(np.ceil(tela[face].max(axis=0)).astype(int), resolucao-1)
        if np.any(maximo < minimo):
            continue
        xx, yy = np.meshgrid(np.arange(minimo[0], maximo[0]+1), np.arange(minimo[1], maximo[1]+1))
        uv = (np.stack((xx, yy), axis=-1)-a) @ np.linalg.inv(matriz_2d).T
        pesos = np.concatenate((1-uv.sum(axis=-1, keepdims=True), uv), axis=-1)
        dentro = np.all(pesos >= -1e-8, axis=-1)
        # Atributos são interpolados com correção de perspectiva.
        pesos_3d = pesos/z if projecao == 'Perspectiva' else pesos
        pesos_3d = pesos_3d / np.maximum(pesos_3d.sum(axis=-1, keepdims=True), 1e-12)
        distancia = np.sum(pesos_3d*z, axis=-1)
        visivel = dentro & (distancia < zbuffer[yy, xx])
        if not np.any(visivel):
            continue
        p = pesos_3d[visivel] @ pontos[face]
        n = normalizar(pesos_3d[visivel] @ normais[face])
        l, v = normalizar(luz-p), normalizar(olho-p)
        nl = np.sum(n*l, axis=-1, keepdims=True)
        r = 2*nl*n-l
        rv = np.maximum(np.sum(r*v, axis=-1, keepdims=True), 0)
        # Phong por pixel: ambiente colorido, difusa colorida e reflexo branco.
        rgb = np.broadcast_to(0.22*cor, p.shape).copy()
        if componente >= 2:
            rgb += 0.72*cor*np.maximum(nl, 0)
        if componente >= 3:
            rgb += ks*0.85*np.where(nl > 0, rv**expoente, 0)
        framebuffer[yy[visivel], xx[visivel]] = np.clip(rgb, 0, 1)
        zbuffer[yy[visivel], xx[visivel]] = distancia[visivel]
    return np.uint8(np.round(255*framebuffer))

##### QUESTÃO1

###### 1.

In [ ]:
def mostrar_geometria(arestas):
    display({'vertices': VERTICES.round(4).tolist(), 'faces': FACES.tolist()})
    mostrar_imagens([renderizar(t=0)], ['Objeto antes da animação'], 'geometria.png')
    if arestas:
        figura = plt.figure(figsize=(6, 6))
        eixo = figura.add_subplot(111, projection='3d')
        eixo.plot_trisurf(*VERTICES.T, triangles=FACES, color='skyblue', edgecolor='black', linewidth=0.3)
        eixo.set(xlabel='X', ylabel='Y', zlabel='Z', title='Vértices conectados pelas faces')
        eixo.set_box_aspect((1.7, 2.3, 1.3))
        plt.show()
        plt.close(figura)


saida_geometria = exibir_controles(mostrar_geometria, {'arestas': widgets.Checkbox(value=True, description='Exibir malha')})

##### QUESTÃO3

###### 1.

In [ ]:
def mostrar_transformacoes(voltas, inclinacao):
    tempos = [0, 0.125, 0.25]
    imagens = [renderizar(t=t, voltas=voltas, inclinacao=inclinacao, ks=0.02, expoente=5) for t in tempos]
    mostrar_imagens(imagens, [f'Rotação Y: {360*voltas*t:.0f}°' for t in tempos], 'transformacoes.png')


saida_transformacoes = exibir_controles(mostrar_transformacoes, {
    'voltas': deslizador(1, 0.25, 2, 0.25, 'Voltas'),
    'inclinacao': deslizador(18, -45, 45, 1, 'Inclinação')})

##### QUESTÃO4

###### 1.

In [ ]:
def mostrar_phong(t, luz_x, luz_y, luz_z, ks, expoente):
    imagens = [renderizar(t=t, componente=c, luz_x=luz_x, luz_y=luz_y, luz_z=luz_z,
                         ks=ks, expoente=expoente) for c in [1, 2, 3]]
    mostrar_imagens(imagens, ['A: ambiente', 'B: ambiente + difusa', 'C: ambiente + difusa + especular'], 'phong.png')


saida_phong = exibir_controles(mostrar_phong, {
    't': deslizador(0.25, 0, 1, 0.01, 'Instante'),
    'luz_x': deslizador(3, -5, 5, 0.25, 'Luz X'),
    'luz_y': deslizador(3, -5, 5, 0.25, 'Luz Y'),
    'luz_z': deslizador(4, 2, 6, 0.25, 'Luz Z'),
    'ks': deslizador(0.95, 0, 1, 0.01, 'ks'),
    'expoente': deslizador(70, 1, 120, 1, 'Expoente')})

##### QUESTÃO5

###### 1.

In [ ]:
controles_material = {
    'ks_inicial': deslizador(0.02, 0, 1, 0.01, 'ks inicial'),
    'ks_final': deslizador(0.95, 0, 1, 0.01, 'ks final'),
    's_inicial': deslizador(5, 1, 120, 1, 's inicial'),
    's_final': deslizador(70, 1, 120, 1, 's final')}


def mostrar_materiais(**parametros):
    materiais = [estado(t, **parametros)[1:] for t in [0, 0.5, 1]]
    imagens = [renderizar(t=0.25, ks=k, expoente=s) for k, s in materiais]
    titulos = [f'{nome}: ks={k:.2f}, s={s:.1f}' for nome, (k, s) in
               zip(['Inicial', 'Intermediário', 'Final'], materiais)]
    mostrar_imagens(imagens, titulos, 'materiais.png')


saida_materiais = exibir_controles(mostrar_materiais, controles_material)

##### QUESTÃO6

###### 1.

In [ ]:
controles_cena = dict(controles_material,
    voltas=deslizador(1, 0.25, 2, 0.25, 'Voltas'),
    camera=deslizador(4.5, 3.5, 7, 0.25, 'Câmera Z'),
    projecao=widgets.Dropdown(options=['Perspectiva', 'Ortográfica'], value='Perspectiva', description='Projeção'),
    luz_movel=widgets.Checkbox(value=False, description='Luz em órbita'))


def mostrar_quadros(**parametros):
    tempos = [0, 0.25, 0.5, 0.75, 1]
    imagens = [renderizar(t=t, **parametros) for t in tempos]
    legendas = []
    for t in tempos:
        angulo, k, s = estado(t, **{chave: valor for chave, valor in parametros.items()
                                  if chave in ['voltas', 'ks_inicial', 'ks_final', 's_inicial', 's_final']})
        aparencia = 'Reflexo fraco' if k < 0.15 else ('Reflexo intermediário' if k < 0.7 else 'Reflexo intenso')
        legendas.append(f'Y={angulo:.0f}°; ks={k:.2f}; s={s:.1f}\n{aparencia}; concentração depende de s')
    mostrar_imagens(imagens, [f'{100*t:.0f}%' for t in tempos], 'quadros.png', legendas)


saida_quadros = exibir_controles(mostrar_quadros, controles_cena)

##### QUESTÃO7

###### 1.

In [ ]:
def explorar_quadro(t, **parametros):
    imagem = renderizar(t=t, **parametros)
    mostrar_imagens([imagem], [f'Prévia — t={t:.2f}'], 'previa.png')


reprodutor = widgets.Play(value=0, min=0, max=100, step=1, interval=80, description='Reproduzir')
instante = deslizador(0, 0, 1, 0.01, 'Tempo')
ligacao_tempo = widgets.dlink((reprodutor, 'value'), (instante, 'value'), transform=lambda valor: valor/100)
display(reprodutor)
saida_previa = exibir_controles(explorar_quadro, dict(controles_cena, t=instante))

###### 2.

In [ ]:
# A primeira execução exporta os defaults. Depois, o botão lê os controles atuais.
# A exportação usa botão para evitar recalcular o GIF inteiro a cada movimento do slider.
fps_animacao = widgets.IntSlider(value=15, min=5, max=30, description='FPS', continuous_update=False)
duracao_animacao = widgets.IntSlider(value=4, min=2, max=8, description='Segundos', continuous_update=False)
resolucao_animacao = widgets.Dropdown(options=[128, 160, 224], value=160, description='Resolução')
botao_animacao = widgets.Button(description='Gerar animação', button_style='primary')
saida_animacao = widgets.Output()
progresso_animacao = widgets.IntProgress(value=0, min=0, max=60, description='Quadros')


def gerar_animacao(evento=None):
    botao_animacao.disabled = True
    try:
        parametros = {nome: controle.value for nome, controle in controles_cena.items()}
        quantidade = fps_animacao.value*duracao_animacao.value
        progresso_animacao.max = quantidade
        progresso_animacao.value = 0
        quadros = []
        pasta_quadros = PASTA / 'quadros'
        pasta_quadros.mkdir(exist_ok=True)
        # Remove somente os quadros antigos desta exportação.
        for antigo in pasta_quadros.glob('quadro_*.png'):
            antigo.unlink()
        for indice, t in enumerate(np.linspace(0, 1, quantidade)):
            imagem = Image.fromarray(renderizar(t=t, resolucao=resolucao_animacao.value, **parametros))
            imagem.save(pasta_quadros / f'quadro_{indice:04d}.png')
            quadros.append(imagem)
            progresso_animacao.value = indice+1
        arquivo = PASTA / 'vinheta.gif'
        # Sem repetição automática: o final brilhante permanece distinto do começo fosco.
        quadros[0].save(arquivo, save_all=True, append_images=quadros[1:],
                       duration=round(1000/fps_animacao.value), disposal=2)
        display(ImagemNotebook(data=arquivo.read_bytes(), format='gif'))
        display(FileLink(str(arquivo)))
    finally:
        botao_animacao.disabled = False


def atualizar_animacao(evento):
    with saida_animacao:
        saida_animacao.clear_output(wait=True)
        gerar_animacao()


botao_animacao.on_click(atualizar_animacao)
display(widgets.VBox([fps_animacao, duracao_animacao, resolucao_animacao,
                      botao_animacao, progresso_animacao]), saida_animacao)
gerar_animacao()

##### QUESTÃO8

###### 1.

In [ ]:
def mostrar_desafio(t):
    imagens = [renderizar(t=t, luz_movel=m) for m in [False, True]]
    mostrar_imagens(imagens, ['Fonte fixa', 'Fonte em órbita'], 'desafio.png')


saida_desafio = exibir_controles(mostrar_desafio, {'t': deslizador(0.25, 0, 1, 0.01, 'Instante')})